In [1]:
import os
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error

print("==================================================")
print("INIT EXP04: LOG-TRANSFORM & CALENDAR PHYSICS")
print("==================================================")

# 1. Load Data
data_dir = r'c:\my-project\kaggle-comp\datajointsxinspire\dataset'
train = pd.read_csv(os.path.join(data_dir, 'train.csv'))
test = pd.read_csv(os.path.join(data_dir, 'test.csv'))
test_hist = pd.read_csv(os.path.join(data_dir, 'test_history.csv'))
movies = pd.read_csv(os.path.join(data_dir, 'movies.csv'))
holidays = pd.read_csv(os.path.join(data_dir, 'holidays.csv'))

print(f"Loaded Train: {train.shape}, Test: {test.shape}")

train['date_show'] = pd.to_datetime(train['date_show'])
test['date_show'] = pd.to_datetime(test['date_show'])
test_hist['date_show'] = pd.to_datetime(test_hist['date_show'])
holidays['date_show'] = pd.to_datetime(holidays['date'])
holidays = holidays[['date_show', 'holiday_tipe', 'day_tipe']]

# 2. Extract D1-D3 Sp Profile
def extract_d1_d3_profile(df_hist):
    df_hist = df_hist.sort_values(['movie_title', 'cinema_ids', 'date_show']).copy()
    first_dates = df_hist.groupby(['movie_title', 'cinema_ids'])['date_show'].transform('min')
    df_hist['day_since_release'] = (df_hist['date_show'] - first_dates).dt.days
    d1_d3 = df_hist[df_hist['day_since_release'].isin([0, 1, 2])]
    sp = d1_d3.groupby(['movie_title', 'cinema_ids']).agg(Sp=('total_ticket', 'mean')).reset_index()
    
    piv = d1_d3.pivot_table(index=['movie_title', 'cinema_ids'], columns='day_since_release', values='total_ticket', fill_value=0).reset_index()
    piv.columns = ['movie_title', 'cinema_ids'] + [f't_d{c+1}' for c in piv.columns[2:]]
    for c in ['t_d1', 't_d2', 't_d3']:
        if c not in piv.columns: piv[c] = 0
    sp = sp.merge(piv, on=['movie_title', 'cinema_ids'], how='left')
    
    for c in ['t_d1', 't_d2', 't_d3']:
        sp[c] = sp[c].fillna(sp['Sp'])
        
    sp['trend_31'] = sp['t_d3'] - sp['t_d1']
    sp['ratio_31'] = (sp['t_d3'] + 1) / (sp['t_d1'] + 1)
    sp['ratio_32'] = (sp['t_d3'] + 1) / (sp['t_d2'] + 1)
    
    return sp

train_sp = extract_d1_d3_profile(train)
test_sp = extract_d1_d3_profile(test_hist)

# 3. Merge Sp to Train and Test
train = train.merge(train_sp, on=['movie_title', 'cinema_ids'], how='left')
test = test.merge(test_sp, on=['movie_title', 'cinema_ids'], how='left')

train['Sp'] = train['Sp'].fillna(1.0)
test['Sp'] = test['Sp'].fillna(1.0)

# Filter train to only target days (Day 4-10)
first_dates_tr = train.groupby(['movie_title', 'cinema_ids'])['date_show'].transform('min')
train['day_since_release'] = (train['date_show'] - first_dates_tr).dt.days
train = train[train['day_since_release'].between(3, 9)].copy()

first_dates_te = test_hist.groupby(['movie_title', 'cinema_ids'])['date_show'].min().reset_index(name='first_date')
test = test.merge(first_dates_te, on=['movie_title', 'cinema_ids'], how='left')
test['day_since_release'] = (test['date_show'] - test['first_date']).dt.days

# 4. Feature Engineering (Date & Calendar)
def process_dates(df):
    df['day_of_week'] = df['date_show'].dt.dayofweek
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    
    df = df.merge(holidays, on='date_show', how='left')
    df['is_holiday'] = df['holiday_tipe'].isin(['holiday', 'cuti_bersama']).astype(int)
    df['is_weekend_or_holiday'] = ((df['is_weekend'] == 1) | (df['is_holiday'] == 1)).astype(int)
    
    df['log1p_Sp'] = np.log1p(df['Sp'])
    return df

train = process_dates(train)
test = process_dates(test)

# Merge Movies Genres
train = train.merge(movies[['original_title', 'genre']], left_on='movie_title', right_on='original_title', how='left')
test = test.merge(movies[['original_title', 'genre']], left_on='movie_title', right_on='original_title', how='left')

# 5. Features and Monotonic Constraints
features = ['day_since_release', 'is_weekend', 'is_holiday', 'is_weekend_or_holiday', 'log1p_Sp', 'day_of_week', 't_d1', 't_d2', 't_d3', 'trend_31', 'ratio_31', 'ratio_32']
cat_cols = ['day_of_week']

for c in ['cinema_ids', 'genre']:
    all_cats = list(set(train[c].dropna().unique()).union(set(test[c].dropna().unique())))
    train[c] = pd.Categorical(train[c], categories=all_cats)
    test[c] = pd.Categorical(test[c], categories=all_cats)
    features.append(c)
    cat_cols.append(c)

y_train = train['total_ticket']
y_train_scaled = y_train / train['Sp']
groups = train['movie_title']

params = {
    'objective': 'mae',
    'learning_rate': 0.05,
    'n_estimators': 1500,
    'random_state': 42,
    'verbose': -1,
    'colsample_bytree': 0.8,
    'subsample': 0.8,
    'max_depth': 8
}

gkf = GroupKFold(n_splits=5)
oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))

print(f"Train samples (Days 4-10): {len(train)}")
print("Starting 5-Fold Cross Validation...")
for fold, (tr_idx, va_idx) in enumerate(gkf.split(train, y_train_scaled, groups)):
    X_tr, y_tr = train.iloc[tr_idx][features], y_train_scaled.iloc[tr_idx]
    X_va, y_va = train.iloc[va_idx][features], y_train_scaled.iloc[va_idx]
    
    w_tr = train.iloc[tr_idx]['Sp']
    w_va = train.iloc[va_idx]['Sp']
    
    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_tr, y_tr, 
        sample_weight=w_tr,
        eval_set=[(X_va, y_va)],
        eval_sample_weight=[w_va],
        callbacks=[lgb.early_stopping(50, verbose=False)],
        categorical_feature=cat_cols
    )
    
    oof_preds[va_idx] = model.predict(X_va)
    test_preds += model.predict(test[features]) / 5

oof_ticket = oof_preds * train['Sp']
oof_ticket = np.clip(oof_ticket, 0, None)
raw_mae = mean_absolute_error(y_train, oof_ticket)

naive_preds = train['Sp']
naive_mae = mean_absolute_error(y_train, naive_preds)
mase = raw_mae / naive_mae

print("\\n" + "="*50)
print(f"OOF MAE   : {raw_mae:.4f}")
print(f"Naive MAE : {naive_mae:.4f}")
print(f"MASE SCORE: {mase:.4f}")
print("="*50)

test_ticket = test_preds * test['Sp']
test_ticket = np.clip(test_ticket, 0, None)
test_ticket = np.round(test_ticket).astype(int)

sub = pd.DataFrame({
    'id': test['id'],
    'total_ticket': test_ticket
})
sub.to_csv('Exp04_Submission_LogMonotonic.csv', index=False)
print(f"File Exp04_Submission_LogMonotonic.csv created! (Length: {len(sub)})")


INIT EXP04: LOG-TRANSFORM & CALENDAR PHYSICS
Loaded Train: (138959, 7), Test: (72611, 5)
Train samples (Days 4-10): 48351
Starting 5-Fold Cross Validation...


c:\Users\ra07z\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ra07z\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ra07z\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ra07z\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is depreca

\n==================================================
OOF MAE   : 181.0431
Naive MAE : 248.9649
MASE SCORE: 0.7272
File Exp04_Submission_LogMonotonic.csv created! (Length: 72611)
